In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
# first-time use execution
!pip install biopython

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
# INSTALL pymesh!!
# upload the .whl file for installation
# https://github.com/nuvolos-cloud/PyMesh/releases
from google.colab import files
uploaded = files.upload()
## install pymesh to calculate the curvature of the protein surface
!pip install pymesh2-0.3.1-cp310-cp310-linux_x86_64.whl --upgrade

In [ ]:
import pymesh


In [ ]:
!pip install trimesh

In [ ]:
!pip install pyglet

In [ ]:
!pip install pyvista

In [ ]:
!python --version

In [ ]:
!pip install ipyvolume

In [ ]:
!pip install pywavefront

In [ ]:
import pywavefront
import pandas as pd
import numpy as np
import pymesh
from Bio.PDB import PDBParser, PDBIO
from scipy.spatial import Delaunay
import trimesh
import itertools
from scipy.spatial import ConvexHull
import pyvista as pv
from pymol import cmd
import matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:
local_path = '/content/drive/MyDrive/@antibody/antigen_epitope/'
filename = '1a2y_B_A_C.obj'

In [ ]:
mesh = pywavefront.Wavefront(local_path+filename, collect_faces=True)

In [ ]:
obj_vertices = mesh.vertices
len(obj_vertices)

In [ ]:
len(set(obj_vertices))

In [ ]:
obj_coords = np.array([v for v in set(obj_vertices)])
obj_mean_curvature = np.array([np.nan] * len(obj_coords))
obj_guassian_curvature = np.array([np.nan] * len(obj_coords))
obj_curvature = np.array([np.nan] * len(obj_coords))
n_neighbors = 6

In [ ]:
len(obj_coords)

In [ ]:
for idx in range(len(obj_coords)):
  current_coords = obj_coords[idx]
  dist_coords = obj_coords - current_coords

  dists = np.linalg.norm(dist_coords, axis=1)
  sorted_indices = np.argsort(dists)

  nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]
  vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

  cloud = pv.PolyData(vertices)
  triangulated = cloud.delaunay_2d()
  shell = triangulated.extract_geometry()

  extracted_vertices = np.array(shell.points)
  extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
  extracted_faces = np.array([face for face in extracted_faces if 0 in face])

  mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
  mesh, info = pymesh.remove_isolated_vertices(mesh)

  mesh.add_attribute("vertex_mean_curvature")
  H = mesh.get_attribute("vertex_mean_curvature")
  mean_H = H[0]

  mesh.add_attribute("vertex_gaussian_curvature")
  K = mesh.get_attribute("vertex_gaussian_curvature")
  gaussian_K = K[0]

  elem = np.square(mean_H) - gaussian_K
  flag = True

  if gaussian_K > 0 and mean_H < 0:
    flag = False
  elif gaussian_K < 0 and mean_H > 0:
    flag = False
  elif gaussian_K > 0 and mean_H > 0:
    flag = False

  if elem < 0:
    elem = 1e-8
  elem = np.array(elem)

  k1 = mean_H + np.sqrt(elem)
  k2 = mean_H - np.sqrt(elem)


  si = (k1 + k2) / (k1 - k2)
  si = np.arctan(si) * (-2 / np.pi)

  if flag:
    si = -si

  obj_mean_curvature[idx] = mean_H
  obj_guassian_curvature[idx] = gaussian_K
  obj_curvature[idx] = si

In [ ]:
import ipyvolume as ipv
from matplotlib import cm

colors = cm.RdBu_r((obj_curvature + 1) / 2)

x_min, x_max = obj_coords[:, 0].min(), obj_coords[:, 0].max()
y_min, y_max = obj_coords[:, 1].min(), obj_coords[:, 1].max()
z_min, z_max = obj_coords[:, 2].min(), obj_coords[:, 2].max()

center = obj_coords.mean(axis=0)
centered_coords = obj_coords - center

ipv.figure(width=600, height=500)
scatter = ipv.scatter(centered_coords[:, 0], centered_coords[:, 1], centered_coords[:, 2], size=100, color=colors[:, :3], marker='sphere')
ipv.xlim(x_min, x_max)
ipv.ylim(y_min, y_max)
ipv.zlim(z_min, z_max)

ipv.show()


In [ ]:
for idx in range(len(obj_coords)):
  current_coords = obj_coords[idx]
  dist_coords = obj_coords - current_coords

  dists = np.linalg.norm(dist_coords, axis=1)
  sorted_indices = np.argsort(dists)

  nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]
  vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

  cloud = pv.PolyData(vertices)
  triangulated = cloud.delaunay_2d()
  shell = triangulated.extract_geometry()

  extracted_vertices = np.array(shell.points)
  extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
  extracted_faces = np.array([face for face in extracted_faces if 0 in face])

  mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
  mesh, info = pymesh.remove_isolated_vertices(mesh)

  mesh.add_attribute("vertex_mean_curvature")
  H = mesh.get_attribute("vertex_mean_curvature")
  mean_H = H[0]

  mesh.add_attribute("vertex_gaussian_curvature")
  K = mesh.get_attribute("vertex_gaussian_curvature")
  gaussian_K = K[0]

  elem = np.square(mean_H) - gaussian_K
  flag = True

  if gaussian_K > 0 and mean_H < 0:
    flag = False

  if elem < 0:
    elem = 1e-8
  elem = np.array(elem)

  k1 = mean_H + np.sqrt(elem)
  k2 = mean_H - np.sqrt(elem)

  si = (k1 + k2) / (k1 - k2)
  si = np.arctan(si) * (-2 / np.pi)

  if flag:
    si = -si

  obj_mean_curvature[idx] = mean_H
  obj_guassian_curvature[idx] = gaussian_K
  obj_curvature[idx] = si

In [ ]:
import ipyvolume as ipv
from matplotlib import cm

colors = cm.RdBu_r((obj_curvature + 1) / 2)

x_min, x_max = obj_coords[:, 0].min(), obj_coords[:, 0].max()
y_min, y_max = obj_coords[:, 1].min(), obj_coords[:, 1].max()
z_min, z_max = obj_coords[:, 2].min(), obj_coords[:, 2].max()

center = obj_coords.mean(axis=0)
centered_coords = obj_coords - center

ipv.figure(width=600, height=500)
scatter = ipv.scatter(centered_coords[:, 0], centered_coords[:, 1], centered_coords[:, 2], size=100, color=colors[:, :3], marker='sphere')
ipv.xlim(x_min, x_max)
ipv.ylim(y_min, y_max)
ipv.zlim(z_min, z_max)

ipv.show()

In [ ]:
for idx in range(len(obj_coords)):
  current_coords = obj_coords[idx]
  dist_coords = obj_coords - current_coords

  dists = np.linalg.norm(dist_coords, axis=1)
  sorted_indices = np.argsort(dists)

  nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]
  vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

  cloud = pv.PolyData(vertices)
  triangulated = cloud.delaunay_2d()
  shell = triangulated.extract_geometry()

  extracted_vertices = np.array(shell.points)
  extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
  extracted_faces = np.array([face for face in extracted_faces if 0 in face])

  mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
  mesh, info = pymesh.remove_isolated_vertices(mesh)

  mesh.add_attribute("vertex_mean_curvature")
  H = mesh.get_attribute("vertex_mean_curvature")
  mean_H = H[0]

  mesh.add_attribute("vertex_gaussian_curvature")
  K = mesh.get_attribute("vertex_gaussian_curvature")
  gaussian_K = K[0]

  elem = np.square(mean_H) - gaussian_K
  # flag = True

  if elem < 0:
    elem = 1e-8
  elem = np.array(elem)

  k1 = mean_H + np.sqrt(elem)
  k2 = mean_H - np.sqrt(elem)

  si = (k1 + k2) / (k1 - k2)
  si = np.arctan(si) * (-2 / np.pi)

  # if flag:
  #   si = -si

  obj_mean_curvature[idx] = mean_H
  obj_guassian_curvature[idx] = gaussian_K
  obj_curvature[idx] = si

In [ ]:
import ipyvolume as ipv
from matplotlib import cm

colors = cm.RdBu_r((obj_curvature + 1) / 2)

x_min, x_max = obj_coords[:, 0].min(), obj_coords[:, 0].max()
y_min, y_max = obj_coords[:, 1].min(), obj_coords[:, 1].max()
z_min, z_max = obj_coords[:, 2].min(), obj_coords[:, 2].max()

center = obj_coords.mean(axis=0)
centered_coords = obj_coords - center

ipv.figure(width=600, height=500)
scatter = ipv.scatter(centered_coords[:, 0], centered_coords[:, 1], centered_coords[:, 2], size=100, color=colors[:, :3], marker='sphere')
ipv.xlim(x_min, x_max)
ipv.ylim(y_min, y_max)
ipv.zlim(z_min, z_max)

ipv.show()

In [ ]:
counter = 0

In [ ]:
#@title TEST generate mesh for all the atoms of pdb
local_path = 'drive/MyDrive/antigen/'

def calculate_curvature(pdb_file_path, n_neighbors, distance_threshold = 3.5):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file_path)
    model = structure[0]
    chain = model['C']

    atoms = [atom for atom in chain.get_atoms()]
    curvatures = [None] * len(atoms)

    # Remove HOH Atom
    atoms = [atom for atom in chain.get_atoms() if atom.get_parent().resname != 'HOH']

    # atoms = [atom for atom in chain.get_atoms()]
    coords = np.array([atom.coord for atom in atoms])
    # coords = coords[:, [2, 0, 1]]
    # print(np.sort(coords,axis=0))
    # return

    # print(len(atoms))
    # print(atoms)
    # return

    temp_counter = 0
    for idx, atom in enumerate(atoms):
        atom_coords = np.array(coords[idx])
        dist_coords = coords - atom_coords

        # Calculate distances and get indices of nearest neighbors
        dists = np.linalg.norm(dist_coords, axis=1)
        sorted_indices = np.argsort(dists)

        nearest_neighbors_idx = sorted_indices[np.logical_and(dists[sorted_indices] > 0, dists[sorted_indices] < distance_threshold)]

        # # Exclude the atom itself by starting from index 1
        # within_threshold_indices = sorted_indices[np.logical_and(dists[sorted_indices] > 0, dists[sorted_indices] <= distance_threshold)]
        # # If there are fewer than n_neighbors within the distance threshold, increase the threshold and repeat
        # while len(within_threshold_indices) < n_neighbors:
        #     distance_threshold += 0.5
        #     within_threshold_indices = sorted_indices[np.logical_and(dists[sorted_indices] > 0, dists[sorted_indices] <= distance_threshold)]
        # nearest_neighbors_idx = within_threshold_indices[:n_neighbors]


        # Create vertices and faces
        vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

        # --------------------------------------------- #
        cloud = pv.PolyData(vertices)
        triangulated = cloud.delaunay_2d()
        shell = triangulated.extract_geometry()

        # shell.save(f"triangulated_{temp_counter}.ply")

        extracted_vertices = np.array(shell.points)
        extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
        extracted_faces = np.array([face for face in extracted_faces if 0 in face])


        vertex_count = defaultdict(int)
        for face in extracted_faces:
            if 0 in face:
                vertices = face[face != 0]
                for vertex in vertices:
                    vertex_count[vertex] += 1
        single_occurrences = [vertex for vertex, count in vertex_count.items() if count == 1]

        if len(single_occurrences) == 2:
          new_face = single_occurrences + [0]
          extracted_faces = np.vstack([extracted_faces, new_face])

        # print(extracted_vertices)
        # print(extracted_faces)
        # print('---------------------------------')
        # --------------------------------------------- #

        # Create mesh
        mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
        mesh, info = pymesh.remove_isolated_vertices(mesh)

        # print(mesh.vertices)
        # print('------------------------------')

        lambda_factor = 0.3
        mu_factor = -1.06 * lambda_factor
        vertices_new = mesh.vertices.copy()
        for _ in range(30):
            laplacian = np.zeros_like(vertices_new)
            for face in mesh.faces:
                for i, vertex in enumerate(face):
                    adj_vertices = face[np.arange(len(face)) != i]
                    laplacian[vertex] += vertices_new[adj_vertices].mean(axis=0)

            movement = laplacian - vertices_new
            vertices_new += lambda_factor * movement
            vertices_new += mu_factor * movement

            # Recreate the mesh after smoothing
            mesh = pymesh.form_mesh(vertices_new, mesh.faces)
            mesh, info = pymesh.remove_isolated_vertices(mesh)

        # # --------------------------------------------- #
        # temp_vertices = mesh.vertices
        # temp_faces = mesh.faces

        # print(temp_vertices)
        # print(temp_faces)
        # print(temp_counter)

        # trimesh_mesh = trimesh.Trimesh(vertices=extracted_vertices, faces=extracted_faces)
        # vertex_colors = np.array([[255, 0, 0] if i == 0 else [0, 0, 255] for i in range(len(trimesh_mesh.vertices))])
        # trimesh_mesh.visual.vertex_colors = vertex_colors
        # trimesh_mesh.export(f"n_{n_neighbors}_output_mesh_{temp_counter}.ply")
        # print(trimesh_mesh.vertex_attributes['mean_curvature'][0])
        # return

        # temp_counter += 1
        # if temp_counter == 5:
        #   print('---- return ----')
        #   return

        # --------------------------------------------- #

        mesh.add_attribute("vertex_mean_curvature")
        H = mesh.get_attribute("vertex_mean_curvature")
        mean_H = H[0]

        mesh.add_attribute("vertex_gaussian_curvature")
        K = mesh.get_attribute("vertex_gaussian_curvature")
        K = K[0]

        elem = np.square(mean_H) - K
        if elem < 0:
          elem = 1e-8
        elem = np.array(elem)

        k1 = K + np.sqrt(elem)
        k2 = K - np.sqrt(elem)

        si = (k1 + k2) / (k1 - k2)
        si = np.arctan(si) * (2 / np.pi)

        curvatures[idx] = si

        # print(curvatures[idx])
        # print('-------------------------------')

    return curvatures

# Function to update B-factor with curvature values
def select_and_update(structure, curvatures):
    for model in structure:
        for chain in model:
            if chain.id == 'C':
                for i, residue in enumerate(chain):
                    for j, atom in enumerate(residue):
                        if i * len(residue) + j < len(curvatures):
                            curvature_value = curvatures[i * len(residue) + j]
                            if curvature_value is not None:
                                atom.set_bfactor(curvature_value)

# Initialize a PDB parser
parser = PDBParser(QUIET=True)
filename = '1a2y_B_A_C.pdb'
# Load the structure
pdb_file_path = local_path + filename
structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)

# Calculate curvatures
n_neighbors_list = [10]
distance_threshold = 9
for n in n_neighbors_list:
    curvatures = calculate_curvature(pdb_file_path, n, distance_threshold)

    # Create a new PDBIO object for saving the modified structure
    io = PDBIO()

    # Save the updated chain C structure
    output_path = f'test_near_{distance_threshold}dis.pdb' # Taubin_test3_{n}
    io.set_structure(structure)
    select_and_update(structure, curvatures)
    io.save(output_path)


In [ ]:
import os
def delete_ply_files_in_current_directory():
    current_directory = os.getcwd()
    for filename in os.listdir(current_directory):
        if filename.endswith('.pdb'):
            os.remove(os.path.join(current_directory, filename))
            print(f"Deleted: {filename}")
delete_ply_files_in_current_directory()

In [ ]:
from numpy.core.fromnumeric import sort
import pandas as pd
import numpy as np
import pymesh
from Bio.PDB import PDBParser, PDBIO

pdb_file_path = local_path + '1a2y_B_A_C.pdb'
parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein', pdb_file_path)
model = structure[0]
chain = model['C']
atoms = [atom for atom in chain.get_atoms()]
coords = np.array([atom.coord for atom in atoms])
curvatures = [None] * len(atoms)
n_neighbors = 5
for idx, atom in enumerate(atoms[67:68]):
  atom_coords = np.array(coords[idx])

  dist_coords = coords - atom_coords
  # Calculate distances and get indices of nearest neighbors
  dists = np.linalg.norm(dist_coords, axis=1)
  nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]

  # Create vertices and faces
  vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]
  faces = [[0, i, i + 1] for i in range(1, n_neighbors)] + [[0, n_neighbors, 1]]


  faces = np.array(faces)

  # Create mesh
  mesh = pymesh.form_mesh(vertices, faces)

  mesh.add_attribute("vertex_mean_curvature")
  H = mesh.get_attribute("vertex_mean_curvature")
  mean_H = H[0]

  mesh.add_attribute("vertex_gaussian_curvature")
  K = mesh.get_attribute("vertex_gaussian_curvature")
  K = K[0]

  if K < 0:
    K = 1e-8


  k1 = mean_H + np.sqrt(K)
  k2 = mean_H - np.sqrt(K)
  si = (k1 + k2) / (k1 - k2)
  si = np.arctan(si) * (2 / np.pi)

  curvatures[idx] = si
  print(vertices)
  print(si)


In [ ]:
#@title TEST Method_2 pymol generate mesh object and then
from pymol import cmd

In [ ]:
from pymol import cmd



pdb_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C.pdb'
obj_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/model_chain_c.obj'  # Replace with your desired output path

# Load the PDB file into an object named 'protein'
cmd.load(pdb_file_path, 'protein')

# Create a selection for only the CA atoms of chain C
cmd.select('chain_c', 'chain C and polymer')

# Show the surface for the selected CA atoms
cmd.show('surface', 'chain_c')

# Save the view as a .png file
cmd.save(obj_file_path, 'chain C', format='obj')




In [ ]:
import pymesh
import numpy as np

# Load mesh
mesh = pymesh.load_mesh(obj_file_path)

# Enable attributes for curvature calculations
mesh.add_attribute("vertex_mean_curvature")
mesh.add_attribute("vertex_gaussian_curvature")

# Retrieve the curvature attributes
H = mesh.get_attribute("vertex_mean_curvature")
K = mesh.get_attribute("vertex_gaussian_curvature")

# Calculate mean curvature (H) and Gaussian curvature (K)
mean_H = np.mean(H)
mean_K = np.mean(K)

# Calculate principal curvatures k1 and k2
elem = np.square(mean_H) - mean_K
elem = np.array(elem)
elem[elem < 0] = 1e-8  # Handle negative values
k1 = mean_H + np.sqrt(elem)
k2 = mean_H - np.sqrt(elem)

# Compute the shape index for each vertex
si = (2/np.pi) * np.arctan((k1 + k2) / (k1 - k2))

# Initialize curvatures list
curvatures = np.zeros(mesh.num_vertices)

# Since we used the mean curvature and mean Gaussian curvature, the shape index would be the same for all vertices.
# Therefore, we populate the 'curvatures' list with the same shape index value.
curvatures.fill(si)

print(f"Curvatures: {curvatures}")


In [ ]:
len(H)

In [ ]:
# visualize the obj file
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interactive

def read_obj(filename):
    vertices = []
    faces = []
    with open(filename, 'r') as f:
        for line in f:
            parts = line.strip().split(" ")
            if parts[0] == "v":
                vertices.append([float(v) for v in parts[1:]])
            elif parts[0] == "f":
                faces.append([int(f.split('/')[0]) for f in parts[1:]])
    return np.array(vertices), np.array(faces)

# Replace with your .obj file path
obj_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/model_chain_c.obj'
vertices, faces = read_obj(obj_file_path)

def plot_figure(elev, azim):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.view_init(elev=elev, azim=azim)

    for face in faces:
        for i in range(len(face)):
            ax.plot([vertices[face[i] - 1][0], vertices[face[i - 1] - 1][0]],
                    [vertices[face[i] - 1][1], vertices[face[i - 1] - 1][1]],
                    [vertices[face[i] - 1][2], vertices[face[i - 1] - 1][2]], 'b-')
    plt.show()

interactive_plot = interactive(plot_figure, elev=widgets.IntSlider(min=0, max=180, step=10, value=30), azim=widgets.IntSlider(min=0, max=360, step=10, value=30))
output = interactive_plot.children[-1]
interactive_plot


In [ ]:
#@title Read the data frame from csv file
import os

save_dir = './drive/MyDrive/antigen/'
df = pd.read_csv(os.path.join(save_dir, '1a2y_B_A_C_info.tsv'),delimiter='\t')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
#@title curvature from info.dat
# test the curvature calculation on a single protein
import pymesh

# Filter out surface residues
surface_residues = df[df['SaSaRate'] > 0.2]

# Initialize an empty list to store curvature
curvatures = [None] * len(df)

# Loop over each surface residue to find its 3 nearest neighbors

n_neighbors = 10

temp_counter = 0

for idx, residue in surface_residues.iterrows():
    coords = df[['SCx', 'SCy', 'SCz']].values.astype(float)
    residue_coords = np.array(residue[['SCx', 'SCy', 'SCz']].values, dtype=float)

    dist_coords = coords - residue_coords

    dists = np.linalg.norm(dist_coords, axis=1)
    nearest_neighbors_idx = np.argsort(dists)[1:n_neighbors+1]  # Exclude itself

    # Create vertices (coordinates of the residue and its neighbors)
    vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

    # Create faces (assuming the points form a tetrahedron)
    # faces = np.array([[0, 1, 2], [0, 2, 3], [0, 3, 1], [1, 2, 3]])
    # faces = [[0, i, i + 1] for i in range(1, n_neighbors)] + [[0, n_neighbors, 1]]
    # faces = np.array(faces)

    cloud = pv.PolyData(vertices)
    triangulated = cloud.delaunay_2d()
    shell = triangulated.extract_geometry()

    # shell.save(f"triangulated_{temp_counter}.ply")
    extracted_vertices = np.array(shell.points)
    extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
    extracted_faces = np.array([face for face in extracted_faces if 0 in face])

    # Create mesh in PyMesh
    mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)

    # Calculate Gaussian curvature
    # mesh.add_attribute("vertex_gaussian_curvature")
    # curvature = mesh.get_attribute("vertex_gaussian_curvature")

    # Store the average curvature
    # curvatures[idx] = np.mean(curvature)

    #-------------------------------------------------------------------#

    mesh.add_attribute("vertex_mean_curvature")
    H = mesh.get_attribute("vertex_mean_curvature")
    mean_H = H[0]
    # print('mean_H: ', mean_H)

    mesh.add_attribute("vertex_gaussian_curvature")
    K = mesh.get_attribute("vertex_gaussian_curvature")
    K = K[0]
    # print('K[0]: ', K)

    elem = np.square(mean_H) - K
    if elem < 0:
      elem = 1e-8
    elem = np.array(elem)
    # print('elem: ', elem)
    # print('elem: ', np.sqrt(elem))

    k1 = mean_H + np.sqrt(elem)
    k2 = mean_H - np.sqrt(elem)

    # print('k1: ', k1)
    # print('k2: ', k2)

    si = (k1 + k2) / (k1 - k2)
    # print('si1: ', si)

    si = np.arctan(si) * (2 / np.pi)
    # print('si2: ', si)



    # temp_counter += 1
    # if temp_counter == 8:
    #   break

    curvatures[idx] = si

    # print(idx, ' --- ', curvatures[idx])

    # print('----------------------------------')
# Add the curvature column to the DataFrame
df['curvatures'] = curvatures

In [ ]:
df.head(15)['curvatures']

In [ ]:
# Initialize a PDB parser
parser = PDBParser(QUIET=True)

# Load the structure
pdb_file_path = '/content/drive/MyDrive/antigen/1a2y_B_A_C.pdb'
structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)

# Create a dictionary with ResID as key and curvature as value
curvature_dict = dict(zip(df['ResID'], df['curvatures']))

# Iterate through residues and update b-factors
for model in structure:
    for chain in model:
        for residue in chain:
            res_id = residue.id[1]  # Get residue number
            if res_id in curvature_dict:
                curvature_value = curvature_dict[res_id]
                if not pd.isna(curvature_value):  # Check if curvature value is NaN
                    for atom in residue:
                        atom.set_bfactor(curvature_value)

# Save the modified structure to a new file
io = PDBIO()
io.set_structure(structure)
io.save(f'1a2y_B_A_C_modified_{n_neighbors}.pdb')